In [ ]:
!pip install roboflow
!pip install ultralytics
!pip install pytorch_lightning
!pip install torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    print("GPU not available")
    device = torch.device("cpu")

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import os
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import cv2
from getpass import getpass
from ultralytics import YOLO
import pytorch_lightning as pl
from torchvision.models.detection.faster_rcnn import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

In [ ]:
api_key = userdata.get("ROBOFLOW_API_KEY") #for Google colab UI
workspace_slug = userdata.get('ROBOFLOW_WORKSPACE')
cell_slug = userdata.get('ROBOFLOW_PROJECT_CELL')
blood_slug = userdata.get('ROBOFLOW_PROJECT_BLOOD')
egg_slug = userdata.get('ROBOFLOW_PROJECT_EGG')
tomato_slug = userdata.get('ROBOFLOW_PROJECT_TOMATO')



In [ ]:
rf = Roboflow(api_key=api_key) # *---- Add api key to secrect w name ROBOFLOW_API_KEY ----*
project = project = rf.workspace(workspace_slug).project(cell_slug)  # *-------- Need to change this to specific dir --------*
version = project.version(2) #version generated on roboflow universe
cell_yolo_dataset = version.download(
                                model_format="yolo26",
                                location = "./cell_detection/yolo26",
                                overwrite=True) #chose this format for PyTorch compatibality


In [ ]:
print(os.listdir(cell_yolo_dataset.location))
assert "data.yaml" in os.listdir(cell_yolo_dataset.location), "Incorrect data format, download as yolo"

In [ ]:
import time
timestamp = time.time()

model = YOLO("yolo26n.pt")
results = model.train(data=f"{cell_yolo_dataset.location}/data.yaml",
                      epochs=100,
                      imgsz=640,
                      single_cls=True,
                      project="./cell_detection/yolo26",
                      name= f"yolo26_cell_train_{timestamp}")

In [ ]:
# export model

from pathlib import Path

# Define your directory path (use '/content/drive/MyDrive/...' for Google Drive)
dir_path = Path('/content/drive/MyDrive/erdos/2026deeplearning/model')

# Create the directory safely
dir_path.mkdir(parents=True, exist_ok=True)


In [ ]:
# Save your file directly inside your Drive folder
with open('./best_model.pt', 'w') as f, open('best.pt', 'w') as f2:
    f.write("This is permanently saved in my Google Drive!")
    f2.write("This is permanently saved in my Google Drive!")

In [ ]:
# view results plot
from IPython.display import Image, display

dev_plot_path = f"/content/runs/detect/cell_detection/yolo26/yolo26_cell_train_{timestamp}/results.png"
display(Image(filename=dev_plot_path))

In [ ]:
# view performance output in csv
results_csv = pd.read_csv(f"/content/runs/detect/cell_detection/yolo26/yolo26_cell_train_{timestamp}/results.csv")
results_csv.head()

In [ ]:
#plot 1
plt.subplots(nrows = 1, ncols = 2, figsize=(15,5))
plt.subplot(1,2,1)
plt.plot(results_csv["train/box_loss"])
plt.plot(results_csv["val/box_loss"])
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train", "val"])
plt.title("box loss")

#plot 2
plt.subplot(1,2,2)
plt.plot(results_csv["train/cls_loss"])
plt.plot(results_csv["val/cls_loss"])
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train", "val"])
plt.title("cls loss")

plt.show()

In [ ]:
metrics = model.val(data=f"{cell_yolo_dataset.location}/data.yaml")
print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}")
print(f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")


### Blood Cell Detection

In [ ]:
from types import new_class
blood_project = rf.workspace(workspace_slug).project(blood_slug)  # *-------- Need to change this to specific dir --------*

new_version_id = blood_project.generate_version(settings={
    "preprocessing": {
        "auto-orient": True,
        "resize": {
            "width": 640,
            "height": 640,
            "format": "Fit within"
        }
    },
    "augmentation": {}  # Leave empty if you don't want to augment images
})

version = blood_project.version(3) #version generated on roboflow universe
blood_yolo_dataset = version.download(
                                model_format="yolo26",
                                location = f"./blood_detection/yolo26/{new_version_id}",
                                overwrite=True) #chose this format for PyTorch compatibality

In [ ]:
#Evaluate YOLO26 model trained on our cell data on blood cell model

metrics = model.val(data=f"{blood_yolo_dataset.location}/data.yaml",
                    split = "test",
                    plots = True,
                    project = "./blood_detection/yolo26",
                    name = f"yolo26_cell_train_{timestamp}")

print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}")
print(f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Get a list of test images to run prediction on
test_images_dir = Path(blood_yolo_dataset.location) / "test" / "images"
test_image_files = list(test_images_dir.glob("*.jpg"))[:5]  # Get up to 5 images

# 2. Run prediction using your trained model
prediction_results = model(test_image_files)

# 3. Select how many images you want to display in your grid
num_examples = min(len(prediction_results), 4)  # Use 4 to fit perfectly in a 2x2 grid
fig, axes = plt.subplots(2, 1, figsize=(12, 8))  # Creates a 2x2 grid layout
axes = axes.ravel()  # Flattens the grid matrix into a simple list for looping

for i in range(2):
    if i < len(prediction_results):
        result = prediction_results[i]
        # .plot() returns a standard BGR numpy array with your targets drawn on top
        plotted_bgr = result.plot()

        # Convert BGR array (OpenCV style) to RGB array (Matplotlib style)
        plotted_rgb = plotted_bgr[:, :, ::-1]

        # Render the image inside its respective grid cell slot
        axes[i].imshow(plotted_rgb)
        axes[i].axis("off")  # Hides graph axis coordinates
    #     axes[i].set_title(f"Example {i+1}")
    # else:
    #     axes[i].axis("off")  # Hide unused subplots if fewer than 4 examples

plt.tight_layout()
plt.show()

### Tomato Detection

In [ ]:
tomato_slug = userdata.get('ROBOFLOW_PROJECT_TOMATO')
tomato_project = rf.workspace(workspace_slug).project(tomato_slug)  # *-------- Need to change this to specific dir --------*
# new_version_id = tomato_project.generate_version(settings={
#     # "preprocessing": {
#     #     "auto-orient": True,
#     #     "resize": {
#     #         "width": 340,
#     #         "height": 340,
#     #         "format": "Fill (with center crop) in"
#     #     }
#     # },
#     # "augmentation": {}  # Leave empty if you don't want to augment images
# })

version = tomato_project.version(1) #version generated on roboflow universe
tomato_yolo_dataset = version.download(
                                model_format="yolo26",
                                location = "./tomato_detection/yolo26",
                                overwrite=True) #chose this format for PyTorch compatibality


In [ ]:
#Evaluate YOLO26 model trained on our cell data on blood cell model

metrics = model.val(data=f"{tomato_yolo_dataset.location}/data.yaml",
                    split = "test",
                    plots = True,
                    project = "./tomato_detection/yolo26",
                    name = f"yolo26_cell_train_{timestamp}")

print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}")
print(f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")

In [ ]:
# 1. Get a list of test images to run prediction on
test_images_dir = Path(tomato_yolo_dataset.location) / "test" / "images"
test_image_files = list(test_images_dir.glob("*.jpg"))[:10]  # Get up to 5 images

# 2. Run prediction using your trained model
prediction_results = model(test_image_files)

# 3. Select how many images you want to display in your grid
num_examples = min(len(prediction_results), 4)  # Use 4 to fit perfectly in a 2x2 grid
fig, axes = plt.subplots(1, 2, figsize=(6, 4))  # Creates a 2x2 grid layout
axes = axes.ravel()  # Flattens the grid matrix into a simple list for looping

for i in range(2):
    if i < len(prediction_results):
        result = prediction_results[i+1]
        # .plot() returns a standard BGR numpy array with your targets drawn on top
        plotted_bgr = result.plot()

        # Convert BGR array (OpenCV style) to RGB array (Matplotlib style)
        plotted_rgb = plotted_bgr[:, :, ::-1]

        # Render the image inside its respective grid cell slot
        axes[i].imshow(plotted_rgb)
        axes[i].axis("off")  # Hides graph axis coordinates
        # axes[i].set_title(f"Example {i+1}")
    else:
        axes[i].axis("off")  # Hide unused subplots if fewer than 4 examples

plt.tight_layout()
plt.show()

### Detecting Eggs

In [ ]:
egg_slug = userdata.get('ROBOFLOW_PROJECT_EGG')
egg_project = rf.workspace(workspace_slug).project(egg_slug)  # *-------- Need to change this to specific dir --------*
# new_version_id = egg_project.generate_version(settings={
#     "preprocessing": {
#         "auto-orient": True,
#         "resize": {
#             "width": 640,
#             "height": 640,
#             "format": "Fill (with center crop) in"
#         }
#     },
#     "augmentation": {}  # Leave empty if you don't want to augment images
# })

version = egg_project.version(1) #version generated on roboflow universe
egg_yolo_dataset = version.download(
                                model_format="yolo26",
                                location = "./egg_detection/yolo26",
                                overwrite=True) #chose this format for PyTorch compatibality


In [ ]:
#Evaluate YOLO26 model trained on our cell data on blood cell model

metrics = model.val(data=f"{egg_yolo_dataset.location}/data.yaml",
                    split = "val",
                    plots = True,
                    project = "./egg_detection/yolo26",
                    name = f"yolo26_cell_train_{timestamp}")

print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}")
print(f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")

In [ ]:
# 1. Get a list of test images to run prediction on
test_images_dir = Path(egg_yolo_dataset.location) / "valid" / "images"
test_image_files = list(test_images_dir.glob("*.jpg"))[:10]  # Get up to 5 images

# 2. Run prediction using your trained model
prediction_results = model(test_image_files)

# 3. Select how many images you want to display in your grid
num_examples = min(len(prediction_results), 10)  # Use 4 to fit perfectly in a 2x2 grid
fig, axes = plt.subplots(1, 2, figsize=(6, 4))  # Creates a 2x2 grid layout
axes = axes.ravel()  # Flattens the grid matrix into a simple list for looping

for i in range(2):
    if i < len(prediction_results):
        result = prediction_results[i+5]
        # .plot() returns a standard BGR numpy array with your targets drawn on top
        plotted_bgr = result.plot()

        # Convert BGR array (OpenCV style) to RGB array (Matplotlib style)
        plotted_rgb = plotted_bgr[:, :, ::-1]

        # Render the image inside its respective grid cell slot
        axes[i].imshow(plotted_rgb)
        axes[i].axis("off")  # Hides graph axis coordinates
        # axes[i].set_title(f"Example {i+1}")
    else:
        axes[i].axis("off")  # Hide unused subplots if fewer than 4 examples

plt.tight_layout()
plt.show()